# Checkpoint de entrega — treino final (política P1)

Roda **um** ajuste com TODAS as pessoas do MINDS, política `--politica-final ultima` (sem avaliação sobreposta ao treino — só a última época, nunca a melhor numa validação que também treinou), semente e backbone fixados conforme [`docs/politica-modelo-final-2026-09-14.md`](../../docs/politica-modelo-final-2026-09-14.md) — não escolhidos por desempenho.

**Não é** LOSO, não gera relatório de acurácia, não é avaliação independente. O checkpoint resultante (`modelo_final.pt`) ainda precisa de exportação e paridade próprias — não herda a aprovação do piloto M01.


## 1. Ambiente e código

Antes de executar, **commitar/publicar as correções e informar o SHA completo aprovado** em `LIBRAS_COMMIT_FINAL` (variável de ambiente) ou `COMMIT_APROVADO` abaixo. Não usar uma branch nem preencher com o HEAD antigo: o snapshot deve conter as novas guardas. O notebook para se o SHA não for informado. Nenhum push/merge é feito automaticamente.

Variante da etapa 2 (augmentação de domínio): definir `LIBRAS_AUG_DOMINIO=1`, ou no Kaggle trocar a linha por `AUG_DOMINIO = True`. O experimento vira `final-s20260917-augdom-v1`. Fora dessa comparação, manter desligado.

Variante da etapa 3 (clipes externos no treino): definir `LIBRAS_EXTRAS_EXTERNOS=1`, ou no Kaggle trocar a linha por `EXTRAS_EXTERNOS = True`, e anexar o dataset privado com `extras-treino-externo.tar.gz`. O experimento vira `final-s20260917-extras-v1`. Não ligar junto com `AUG_DOMINIO`.

Localmente, usar a variável de ambiente para não modificar este notebook rastreado; no Kaggle, o notebook enviado ao editor é separado do clone. Esta receita exige GPU CUDA. O registro de ambiente permite rastrear a execução, não garante determinismo entre versões/dispositivos.

In [ ]:
import os, pathlib, re, subprocess, sys, tarfile, tempfile, json

EM_KAGGLE = pathlib.Path("/kaggle/working").is_dir() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
BASE = pathlib.Path("/kaggle/working" if EM_KAGGLE else ".").resolve()
URL = "https://github.com/Heitorvazeg/libras-livre-ai-glasses-brasil.git"
# Preencher com o SHA COMPLETO do commit que inclui estas correções, após
# commit/publicação. Não usar uma branch móvel nem inferir HEAD automaticamente.
COMMIT_APROVADO = os.environ.get("LIBRAS_COMMIT_FINAL", "")
# Variante da etapa 2 (docs/etapa2-augmentacao-dominio-protocolo-2026-09-15.md).
# Fora dela, manter desligado: a receita aprovada não usa --aug-dominio.
AUG_DOMINIO = os.environ.get("LIBRAS_AUG_DOMINIO", "0") == "1"
# Variante da etapa 3 (docs/etapa3-treino-multidominio-protocolo-2026-09-15.md).
EXTRAS_EXTERNOS = os.environ.get("LIBRAS_EXTRAS_EXTERNOS", "0") == "1"
if AUG_DOMINIO and EXTRAS_EXTERNOS:
    raise RuntimeError("Etapas 2 e 3 são comparações separadas; ligue só uma variante.")
if not re.fullmatch(r"[0-9a-f]{40}", COMMIT_APROVADO):
    raise RuntimeError("Defina COMMIT_APROVADO (ou LIBRAS_COMMIT_FINAL) com o SHA completo já publicado.")

def git(*args, repo=None):
    cmd = ["git"] + (["-C", str(repo)] if repo else []) + list(args)
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except subprocess.CalledProcessError as erro:
        raise RuntimeError(f"Git falhou: {erro.stderr.strip()}. No Kaggle, habilite Internet e "
                           "confirme que o commit aprovado foi publicado.") from erro

if EM_KAGGLE:
    REPO = BASE / f"libras-final-{COMMIT_APROVADO[:12]}"
    if not REPO.exists():
        git("clone", "--no-checkout", URL, str(REPO))
        git("fetch", "origin", COMMIT_APROVADO, repo=REPO)
        git("checkout", "--detach", COMMIT_APROVADO, repo=REPO)
    # Clone existente nunca recebe reset/merge/checkout implícito.
else:
    REPO = next((p for p in (BASE, *BASE.parents)
                 if (p / "computer-vision-model" / "treino").is_dir()), None)
    if REPO is None:
        raise RuntimeError("Execute localmente dentro do repositório.")

TREINO = (REPO / "computer-vision-model" / "treino").resolve()
if git("rev-parse", "HEAD", repo=REPO) != COMMIT_APROVADO:
    raise RuntimeError("Clone não está no commit aprovado; use um destino novo.")
for nome in ("codigo_final.py", "entrada_final.py", "test_politica_final.py"):
    if not (TREINO / nome).is_file():
        raise RuntimeError(f"Snapshot sem {nome}; publique as correções antes da run.")
sys.path.insert(0, str(TREINO))
import codigo_final
COMMIT_CODIGO = codigo_final.conferir_codigo(REPO, COMMIT_APROVADO)
print("ambiente:", "Kaggle" if EM_KAGGLE else "local")
print("código fixado:", COMMIT_CODIGO, "|", TREINO)

In [ ]:
import importlib.util
faltantes = [pacote for modulo, pacote in (("yaml", "pyyaml"), ("scipy", "scipy"))
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)
import torch, torchvision, numpy, scipy, yaml
if not torch.cuda.is_available():
    raise RuntimeError("Esta receita do notebook exige CUDA; no Kaggle, habilite Accelerator = GPU.")
AMBIENTE = {
    "commit": COMMIT_CODIGO, "python": sys.version, "executavel": sys.executable,
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "numpy": numpy.__version__, "scipy": scipy.__version__, "pyyaml": yaml.__version__,
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0), "threads": 4, "workers": 2,
    "pacotes": subprocess.run([sys.executable, "-m", "pip", "freeze"],
                              capture_output=True, text=True, check=True).stdout.splitlines(),
}
print("torch", torch.__version__, "| GPU:", AMBIENTE["gpu"])

## 2. Insumos: MINDS, backbone aprovado e extras opcionais da etapa 3

O mesmo dataset **privado** do Kaggle pode conter os três corpora em pastas separadas; não é necessário outro upload dos corpora completos:
- `landmarks-minds/landmarks/`: pasta interna com os landmarks MINDS, usada no fine-tuning.
- `landmarks-malta/ladmarks-malta/`: hierarquia informada para MALTA; não é lida diretamente por este notebook.
- `landmarks-vlibrasil/landmarks-pretreino-auditado/`: não é lida diretamente por este notebook.

No Kaggle, a célula seguinte procura `landmarks-minds/landmarks/` dentro dos datasets anexados, sem depender do nome do dataset privado. Se houver mais de uma candidata, informe o caminho completo da **pasta interna** em `ORIGEM_MINDS`. Sem essa hierarquia, permanece a descoberta por `landmarks-minds.tar.gz` ou pasta `landmarks`. Um caminho explícito tem prioridade.

A entrada MINDS selecionada deve conter exatamente **800 clipes: oito pessoas, vinte sinais, reps 01–05**, arrays float32 finitos `(T>=3,57,3)`. Não selecionar uma pasta com clipes MINDS e V-LIBRASIL misturados. Os 30 clipes V da pasta local histórica devem permanecer preservados fora da entrada MINDS.

Também disponibilize o **backbone pré-treinado aprovado**, como `backbone_gcn.pt` ou pacote que o contenha. `BACKBONE_EXPLICITO` aceita o caminho do arquivo de pesos, não do pacote. O hash é conferido antes do uso. Este notebook **não refaz o pré-treino**: carrega os pesos aprendidos com V-LIBRASIL + MALTA. A receita base ajusta somente com MINDS; não usa WLASL/negativos extras.

**Somente na etapa 3:** anexe também `extras-treino-externo.tar.gz`, pacote privado já preparado em `experimentos-privados/etapa3-extras/` (aproximadamente 3,2 MB). A descoberta independe do nome do dataset Kaggle e exige um único pacote com o hash fixado abaixo. São 42 clipes de sete pessoas, 17 classes, repetidos cinco vezes por época; nenhum dos 28 clipes do grupo de avaliação entra nesse pacote. Não substituir pelos corpora completos. Mantenha `EXTRAS_EXTERNOS = True` e `AUG_DOMINIO = False` para esta comparação.

A preparação não altera os insumos: instala em destino exclusivo e só reutiliza uma preparação com identidade e bytes iguais. Opcionalmente fornecer `MANIFESTO_REFERENCIA`, inventário privado validado na origem. Hashes locais não certificam a extração histórica.

In [ ]:
import entrada_pretreino as entrada
import entrada_final

HASH_BACKBONE_APROVADO = "7a6e997c5830139162b32bc9b37a48e6eb5b8d6222836da87cb95e94ecf6baf5"
RAIZ_INPUT = pathlib.Path("/kaggle/input") if EM_KAGGLE else BASE
DESTINO = BASE / "dados-treino-final-v1"
# Opcional: caminho completo da pasta INTERNA landmarks-minds/landmarks/.
# None permite descobrir essa hierarquia dentro do dataset privado no Kaggle.
ORIGEM_MINDS = None
BACKBONE_EXPLICITO = None
# Opcional: inventário privado validado na origem para conferir os mesmos bytes.
MANIFESTO_REFERENCIA = None

if EM_KAGGLE and ORIGEM_MINDS is None:
    pastas_minds = sorted(p for p in RAIZ_INPUT.rglob("landmarks-minds/landmarks")
                          if p.is_dir())
    if len(pastas_minds) > 1:
        raise RuntimeError("Mais de uma pasta MINDS: defina ORIGEM_MINDS com a pasta interna. "
                           "Candidatas: " + ", ".join(map(str, pastas_minds)))
    if pastas_minds:
        ORIGEM_MINDS = pastas_minds[0]

origem_minds = entrada.localizar(RAIZ_INPUT, "minds", ORIGEM_MINDS)
print("Origem MINDS selecionada:", origem_minds)
MINDS = entrada_final.preparar_minds(
    origem_minds, DESTINO / "minds", manifesto_referencia=MANIFESTO_REFERENCIA)
INVENTARIO_MINDS = entrada_final.validar_minds(MINDS)
print("MINDS completo:", INVENTARIO_MINDS["n_clipes"], "clipes |",
      INVENTARIO_MINDS["corpus_sha256"])

# Backbone em arquivo ou pacote; nunca extrair sobre uma pasta já usada.
if BACKBONE_EXPLICITO is not None:
    candidatos = [pathlib.Path(BACKBONE_EXPLICITO).resolve()]
else:
    candidatos = sorted(set(RAIZ_INPUT.rglob("backbone_gcn.pt")))
if len(candidatos) == 1:
    BACKBONE = candidatos[0]
elif candidatos:
    raise RuntimeError("Mais de um backbone: defina BACKBONE_EXPLICITO, sem escolha automática.")
else:
    tars = sorted(p for p in RAIZ_INPUT.rglob("*.tar.gz") if "backbone" in p.name)
    if len(tars) != 1:
        raise RuntimeError("Anexe exatamente um pacote de backbone ou defina BACKBONE_EXPLICITO.")
    DESTINO.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".backbone-", dir=DESTINO) as tmp:
        with tarfile.open(tars[0], "r:gz") as tar:
            membros = tar.getmembers()
            if any(not (m.isdir() or m.isfile()) for m in membros):
                raise RuntimeError("Pacote do backbone contém links ou arquivos irregulares.")
            tar.extractall(tmp, filter="data")
        achados = list(pathlib.Path(tmp).rglob("backbone_gcn.pt"))
        if len(achados) != 1:
            raise RuntimeError("Pacote não contém exatamente um backbone_gcn.pt.")
        bruto = achados[0].read_bytes()
        if entrada.pv.hash_arquivo(achados[0]) != HASH_BACKBONE_APROVADO:
            raise RuntimeError("Backbone do pacote não é o aprovado.")
        BACKBONE = DESTINO / f"backbone-{HASH_BACKBONE_APROVADO}.pt"
        if BACKBONE.exists():
            if BACKBONE.is_symlink() or BACKBONE.read_bytes() != bruto:
                raise RuntimeError("Backbone preparado foi alterado; nada será sobrescrito.")
        else:
            with BACKBONE.open("xb") as arquivo:
                arquivo.write(bruto)

if BACKBONE.is_symlink() or not BACKBONE.is_file():
    raise RuntimeError("Backbone deve ser arquivo regular.")
hash_real = entrada.pv.hash_arquivo(BACKBONE)
if hash_real != HASH_BACKBONE_APROVADO:
    raise RuntimeError(f"Backbone com hash {hash_real}; não usar.")
print("Backbone verificado:", BACKBONE, "|", hash_real)

# Etapa 3: manifesto versionado no clone + pacote privado com os mesmos bytes.
MANIFESTO_EXTRAS = TREINO / "extras_treino_externo_manifesto.json"
HASH_PACOTE_EXTRAS = "2ee1091658ef5080988faa9e222671152328a06e6c1e56768bb6d00bd0aa3652"
RAIZ_EXTRAS, PESSOAS_EXTRAS, ORIGEM_EXTRAS = None, [], None
if EXTRAS_EXTERNOS:
    import extras_externos
    itens_extras = json.loads(MANIFESTO_EXTRAS.read_text(encoding="utf-8"))["itens"]
    RAIZ_EXTRAS = DESTINO / f"extras-{entrada.pv.hash_arquivo(MANIFESTO_EXTRAS)[:12]}"
    # O Kaggle descompacta pacotes enviados: aceitar o pacote OU a pasta extraída, nunca os dois.
    pacotes = sorted(RAIZ_INPUT.rglob("extras-treino-externo.tar.gz"))
    primeiro = pathlib.PurePosixPath(itens_extras[0]["arquivo"])
    pastas = sorted({p.parents[len(primeiro.parts) - 1] for p in RAIZ_INPUT.rglob(primeiro.name)
                     if p.as_posix().endswith("/" + primeiro.as_posix())})
    if len(pacotes) + len(pastas) != 1:
        raise RuntimeError("Anexe exatamente uma cópia dos extras (pacote .tar.gz ou pasta extraída). "
                           f"Pacotes: {pacotes}; pastas: {pastas}")
    if pacotes and entrada.pv.hash_arquivo(pacotes[0]) != HASH_PACOTE_EXTRAS:
        raise RuntimeError("Pacote de extras não é o aprovado.")
    ORIGEM_EXTRAS = ({"forma": "pacote", "sha256": HASH_PACOTE_EXTRAS} if pacotes
                     else {"forma": "pasta_extraida", "caminho": str(pastas[0])})
    if not RAIZ_EXTRAS.exists():
        DESTINO.mkdir(parents=True, exist_ok=True)
        tmp = pathlib.Path(tempfile.mkdtemp(prefix=".extras-", dir=DESTINO))
        if pacotes:
            with tarfile.open(pacotes[0], "r:gz") as tar:
                if any(not (m.isdir() or m.isfile()) for m in tar.getmembers()):
                    raise RuntimeError("Pacote de extras contém links ou arquivos irregulares.")
                tar.extractall(tmp, filter="data")
        else:
            # Só os arquivos do manifesto; extras_externos.ler confere o hash de cada um logo abaixo.
            for item in itens_extras:
                origem_item = pastas[0] / item["arquivo"]
                if not origem_item.is_file():
                    raise RuntimeError(f"Pasta de extras sem {item['arquivo']}.")
                destino_item = tmp / item["arquivo"]
                destino_item.parent.mkdir(parents=True, exist_ok=True)
                destino_item.write_bytes(origem_item.read_bytes())
        tmp.rename(RAIZ_EXTRAS)
    dados_extras, _ = extras_externos.ler(MANIFESTO_EXTRAS, RAIZ_EXTRAS)
    PESSOAS_EXTRAS = dados_extras["pessoas"]
    print("Extras externos:", dados_extras["n_clipes"], "clipes |", ", ".join(PESSOAS_EXTRAS))

## 3. Selftest — obrigatório antes do treino longo


In [ ]:
def executar(script, argumentos, log):
    comando = [sys.executable, "-u", script, *argumentos]
    print("Executando:", " ".join(comando), flush=True)
    with (EXP / log).open("a", encoding="utf-8") as arquivo_log:
        with subprocess.Popen(comando, cwd=TREINO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as processo:
            for linha in processo.stdout:
                print(linha, end="", flush=True)
                arquivo_log.write(linha)
                arquivo_log.flush()
            if processo.wait():
                raise subprocess.CalledProcessError(processo.returncode, comando)

NOME_EXPERIMENTO = ("final-s20260917-augdom-v1" if AUG_DOMINIO else
                    "final-s20260917-extras-v1" if EXTRAS_EXTERNOS else "final-s20260917-v1")
EXP = BASE / "experimentos-privados" / NOME_EXPERIMENTO
EXP.mkdir(parents=True, exist_ok=True)

def registrar(nome, valor):
    caminho = EXP / nome
    texto = json.dumps(valor, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n"
    if caminho.exists():
        if caminho.is_symlink() or caminho.read_text(encoding="utf-8") != texto:
            raise RuntimeError(f"Registro {nome} diverge; use outro experimento, sem sobrescrever.")
    else:
        with caminho.open("x", encoding="utf-8") as f:
            f.write(texto)
    return caminho

registrar("ambiente.json", AMBIENTE)
INVENTARIO_CAMINHO = registrar("inventario-minds.json", INVENTARIO_MINDS)
registrar("inicializacao.json", {"backbone_sha256": hash_real, "commit": COMMIT_CODIGO})
codigo_final.conferir_codigo(REPO, COMMIT_APROVADO)
executar("test_entrada_final.py", [], "preflight.log")
executar("test_codigo_final.py", [], "preflight.log")
executar("test_politica_final.py", [], "preflight.log")
if EXTRAS_EXTERNOS:
    dados_extras, _ = extras_externos.ler(MANIFESTO_EXTRAS, RAIZ_EXTRAS)
    if dados_extras != extras_externos.gerar(TREINO / "calibracao_naovista_manifesto.json"):
        raise RuntimeError("Manifesto de extras diverge do grupo fixado no manifesto de origem.")
    registrar("manifesto-extras.json", dados_extras)
    registrar("entrada-extras.json", {
        "manifesto_sha256": entrada.pv.hash_arquivo(MANIFESTO_EXTRAS),
        "origem_pacote": ORIGEM_EXTRAS,
        "n_clipes": dados_extras["n_clipes"], "repeticoes": 5,
        "pessoas": dados_extras["pessoas"],
        "pessoas_excluidas": dados_extras["origem"]["pessoas_excluidas"],
    })
    executar("test_extras_externos.py", [], "preflight.log")
executar("selftest.py", [], "selftest.log")
print("Testes passaram. Saídas privadas:", EXP)

## 4. O treino final

`--politica-final ultima` exige `--semente` explícita e recusa saída não-vazia — não sobrescreve uma tentativa anterior por engano. Sem avaliação durante o treino: perda/acurácia no log são diagnóstico, não seleção.


In [ ]:
# Revalidar imediatamente antes da chamada longa, inclusive se células foram
# executadas fora de ordem. O próprio CLI repete a guarda antes de criar o modelo.
codigo_final.conferir_codigo(REPO, COMMIT_APROVADO)
entrada_final.validar_minds(MINDS, manifesto_referencia=INVENTARIO_CAMINHO)
if EXTRAS_EXTERNOS:
    extras_externos.ler(MANIFESTO_EXTRAS, RAIZ_EXTRAS)
if entrada.pv.hash_arquivo(BACKBONE) != HASH_BACKBONE_APROVADO:
    raise RuntimeError("Backbone mudou após o preflight.")
SAIDA_FINAL = EXP / "modelo_final"
FINAL_ARGS = [
    "--arquitetura", "gcn", "--ossos", "--com-z", "--z-recentrado",
    "--kernel-temporal", "9", "--fontes", "minds", "--landmarks", str(MINDS),
    "--inventario-final", str(INVENTARIO_CAMINHO), "--inicializar", str(BACKBONE),
    "--epocas", "120", "--lr", "1e-3", "--wd", "1e-4", "--batch", "64",
    "--agendador", "cosseno",
    "--final", "--politica-final", "ultima", "--semente", "20260917",
    "--dispositivo", "cuda", "--threads", "4", "--workers", "2", "--saida", str(SAIDA_FINAL),
]
if AUG_DOMINIO:
    FINAL_ARGS.append("--aug-dominio")
if EXTRAS_EXTERNOS:
    FINAL_ARGS += ["--extras-manifesto", str(MANIFESTO_EXTRAS), "--extras-raiz", str(RAIZ_EXTRAS),
                   "--extras-repeticoes", "5"]
registrar("comando-final.json", FINAL_ARGS)
executar("treinar.py", FINAL_ARGS, "treino-final.log")

destino = SAIDA_FINAL / "modelo_final.pt"
if not destino.is_file():
    raise RuntimeError("treinar.py terminou sem gerar modelo_final.pt.")
# Somente checkpoint próprio, gerado nesta run. Não carregar pickle de terceiros.
checkpoint = torch.load(destino, map_location="cpu", weights_only=False)
meta = checkpoint["meta"]
if (meta["epoca_salva"] != 120 or meta["politica_selecao"] != "ultima"
        or meta["avaliacao_independente"] is not False
        or meta["backbone_sha256"] != HASH_BACKBONE_APROVADO
        or checkpoint["rotulos"] != INVENTARIO_MINDS["rotulos"]
        or sorted(meta["pessoas"]) != sorted(set(INVENTARIO_MINDS["pessoas"]) | set(PESSOAS_EXTRAS))
        or (meta.get("extras") or {}).get("manifesto_sha256") != (
            entrada.pv.hash_arquivo(MANIFESTO_EXTRAS) if EXTRAS_EXTERNOS else None)
        or (meta.get("extras") or {}).get("repeticoes") != (5 if EXTRAS_EXTERNOS else None)
        or meta["proveniencia"]["codigo"]["commit"] != COMMIT_APROVADO
        or bool(meta.get("args", {}).get("aug_dominio", False)) != AUG_DOMINIO
        or any(not torch.isfinite(v).all() for v in checkpoint["state_dict"].values())):
    raise RuntimeError("Checkpoint não corresponde à política final; não exportar.")
registrar("checkpoint-final.json", {"sha256": entrada.pv.hash_arquivo(destino),
                                   "epoca": 120, "aprovado_entrega": False})
del checkpoint
print("Checkpoint final:", destino, "|", entrada.pv.hash_arquivo(destino))

## 5. Backup privado


In [ ]:
def empacotar():
    arquivo = EXP.parent / f"{EXP.name}.tar.gz"
    parcial = arquivo.with_suffix(".parcial")
    with tarfile.open(parcial, "w:gz") as tar:
        tar.add(EXP, arcname=EXP.name)
    parcial.replace(arquivo)
    return arquivo

arquivo = empacotar()
print("Arquivo privado:", arquivo, "|", arquivo.stat().st_size, "bytes")
print("Kaggle: Save Version e mantenha notebook, datasets e outputs PRIVADOS.")
